# Build stock and news dataframes

Run after `collect_data.ipynb` and `clean_data.ipynb`. Following the EDA, use AAPL and JNJ. Keep the daily stock fields and add close lags for the previous one, two, and three trading days per ticker. Each lag is missing where the ticker has insufficient history.

Match articles to the ticker and publication date in New York time. Average each FinBERT probability across articles on the same ticker-day. Articles on dates without a matching trading bar are omitted. The publication date includes after-close articles, so these scores represent the full calendar day.

In [4]:
import os
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

os.environ["USE_TF"] = "0"
from transformers import pipeline

data_dir = Path("../data") if Path.cwd().name == "src" else Path("data")
prices = pd.read_parquet(data_dir / "raw/prices.parquet")
articles = pd.read_parquet(data_dir / "processed/news_clean.parquet")

df_original = prices.loc[prices["symbol"].isin(["AAPL", "JNJ"])].copy()
df_original["timestamp"] = pd.to_datetime(df_original["timestamp"], utc=True)
df_original = df_original.sort_values(["symbol", "timestamp"]).reset_index(drop=True)
close_by_symbol = df_original.groupby("symbol")["close"]
for lag in range(1, 4):
    df_original[f"close_lag_{lag}"] = close_by_symbol.shift(lag)
df_original.head()

,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,close_lag_1,close_lag_2,close_lag_3
0,AAPL,2020-01-02 05:00:00+00:00,71.23,72.28,70.98,72.22,146217352.0,283489.0,71.78,NaN,NaN,NaN
1,AAPL,2020-01-03 05:00:00+00:00,71.45,72.27,71.29,71.51,154711256.0,280441.0,71.85,72.22,NaN,NaN
2,AAPL,2020-01-06 05:00:00+00:00,70.64,72.12,70.39,72.08,125599968.0,242868.0,71.61,71.51,72.22,NaN
3,AAPL,2020-01-07 05:00:00+00:00,72.09,72.35,71.53,71.75,118227136.0,227871.0,71.90,72.08,71.51,72.22
4,AAPL,2020-01-08 05:00:00+00:00,71.45,73.20,71.45,72.90,142736804.0,261773.0,72.45,71.75,72.08,71.51


In [5]:
labels = ["negative", "neutral", "positive"]
keys = ["symbol", "timestamp"]

articles = articles.rename(columns={"ticker": "symbol"})
articles["timestamp"] = (
    pd.to_datetime(articles["created_at"], utc=True)
    .dt.tz_convert("America/New_York")
    .dt.normalize()
    .dt.tz_convert("UTC")
    .astype(df_original["timestamp"].dtype)
)
articles = articles.merge(df_original[keys], on=keys, validate="many_to_one")
articles["text"] = articles["text_for_sentiment"].fillna("").map(
    lambda text: BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
)
articles = articles.loc[articles["text"].ne("")].reset_index(drop=True)

scores = pd.DataFrame(index=articles.index, columns=labels, dtype=float)
if not articles.empty:
    sentiment = pipeline(
        "text-classification", model="ProsusAI/finbert", framework="pt", device="cpu"
    )
    results = sentiment(
        articles["text"].tolist(), truncation=True, top_k=None, batch_size=8
    )
    scores = pd.DataFrame(
        [{item["label"]: item["score"] for item in result} for result in results]
    )[labels]

daily_sentiment = articles[keys].join(scores).groupby(keys)[labels].mean()
df_news = df_original.join(daily_sentiment, on=keys, validate="one_to_one")
df_news[labels] = df_news[labels].fillna(0.0)

pd.testing.assert_frame_equal(df_news[df_original.columns], df_original)
has_news = pd.MultiIndex.from_frame(df_original[keys]).isin(daily_sentiment.index)
assert df_news.loc[~has_news, labels].eq(0.0).all().all()
assert df_news.loc[has_news, labels].sum(axis=1).sub(1.0).abs().lt(1e-5).all()
df_news.head()

/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap,close_lag_1,close_lag_2,close_lag_3,negative,neutral,positive
0,AAPL,2020-01-02 05:00:00+00:00,71.23,72.28,70.98,72.22,146217352.0,283489.0,71.78,NaN,NaN,NaN,0.0,0.0,0.0
1,AAPL,2020-01-03 05:00:00+00:00,71.45,72.27,71.29,71.51,154711256.0,280441.0,71.85,72.22,NaN,NaN,0.0,0.0,0.0
2,AAPL,2020-01-06 05:00:00+00:00,70.64,72.12,70.39,72.08,125599968.0,242868.0,71.61,71.51,72.22,NaN,0.0,0.0,0.0
3,AAPL,2020-01-07 05:00:00+00:00,72.09,72.35,71.53,71.75,118227136.0,227871.0,71.90,72.08,71.51,72.22,0.0,0.0,0.0
4,AAPL,2020-01-08 05:00:00+00:00,71.45,73.20,71.45,72.90,142736804.0,261773.0,72.45,71.75,72.08,71.51,0.0,0.0,0.0


## Verify columns and inspect sample values

In [6]:
expected_stock_columns = [
    "symbol", "timestamp", "open", "high", "low", "close", "volume",
    "trade_count", "vwap", "close_lag_1", "close_lag_2", "close_lag_3",
]
assert df_original.columns.tolist() == expected_stock_columns
assert df_news.columns.tolist() == expected_stock_columns + ["negative", "neutral", "positive"]

for lag in (1, 2, 3):
    expected_lag = df_original.groupby("symbol")["close"].shift(lag)
    pd.testing.assert_series_equal(df_original[f"close_lag_{lag}"], expected_lag, check_names=False)
assert df_news[labels].ge(0).all().all() and df_news[labels].le(1).all().all()
news_rows = df_news.set_index(keys).loc[daily_sentiment.index, labels]
pd.testing.assert_frame_equal(news_rows, daily_sentiment)

for name, frame in [("df_original", df_original), ("df_news", df_news)]:
    print(f"{name} ({len(frame):,} rows): {frame.columns.tolist()}")

sample_columns = keys + ["close", "close_lag_1", "close_lag_2", "close_lag_3"] + labels
print(f"With scored news ({has_news.sum():,} rows)")
display(df_news.loc[has_news, sample_columns].head(3))
print(f"Without scored news ({(~has_news).sum():,} rows)")
display(df_news.loc[~has_news, sample_columns].head(3))
print("All dataframe checks passed.")

df_original (3,412 rows): ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'close_lag_1', 'close_lag_2', 'close_lag_3']
df_news (3,412 rows): ['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap', 'close_lag_1', 'close_lag_2', 'close_lag_3', 'negative', 'neutral', 'positive']
With scored news (198 rows)


,symbol,timestamp,close,close_lag_1,close_lag_2,close_lag_3,negative,neutral,positive
1588,AAPL,2026-04-29 04:00:00+00:00,269.48,270.02,266.93,270.37,0.188503,0.428723,0.382774
1589,AAPL,2026-04-30 04:00:00+00:00,270.66,269.48,270.02,266.93,0.016698,0.362364,0.620938
1590,AAPL,2026-05-01 04:00:00+00:00,279.43,270.66,269.48,270.02,0.304058,0.132244,0.563699


Without scored news (3,214 rows)


,symbol,timestamp,close,close_lag_1,close_lag_2,close_lag_3,negative,neutral,positive
0,AAPL,2020-01-02 05:00:00+00:00,72.22,NaN,NaN,NaN,0.0,0.0,0.0
1,AAPL,2020-01-03 05:00:00+00:00,71.51,72.22,NaN,NaN,0.0,0.0,0.0
2,AAPL,2020-01-06 05:00:00+00:00,72.08,71.51,72.22,NaN,0.0,0.0,0.0


All dataframe checks passed.


In [9]:
print(df_original.columns)
print(df_news.columns)

TypeError: 'Index' object is not callable